# 8-bit attention (SageAttention2) on top of compile: Wan2.1-T2V-1.3B on L4

Runtime: L4. Predictions to check: `predictions_sage_2026-11-11.md` (S1–S8). All runs share this session.

Time: about 10–15 min to build SageAttention, then about 75 min of runs (~3 compute units). The build cell stops the notebook if SageAttention does not import, so a failed build costs only those minutes. The last section (fp16-PV variant) is optional.

Second attempt (2026-11-12). The first run's build failed and the notebook kept going without Sage. diffusers is now pinned to 0.42.1, the version of the stored hashes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit --format=csv

In [ ]:
!pip install -q diffusers==0.42.1 transformers==5.22.2 accelerate ftfy imageio imageio-ffmpeg lpips 2>&1 | tail -5
import torch, diffusers, transformers
print(torch.__version__, 'cuda', torch.version.cuda, diffusers.__version__, transformers.__version__)
!nvcc --version | tail -2

In [ ]:
%cd /content
!git -C video-dit-l4 pull || git clone https://github.com/FredWen-S/video-dit-l4.git
%cd /content/video-dit-l4

## Build SageAttention for sm_89 (~10–15 min)

torch's extension builder needs nvcc with the same major version as torch's CUDA. If they differ, the cell installs the matching nvcc and headers from NVIDIA's apt repository. The full build log goes to `/content/sage_build.log`; the cell prints its last 40 lines and raises an error if `sageattention` does not import. If it fails, send me those lines.

Attempt 2 failed here with `'sizes' is not a member of 'at::symint'`: torch 2.14's headers need C++20, and SageAttention's `setup.py` compiles with `-std=c++17`. The cell now switches both flag lists to C++20.

In [ ]:
import os, re, subprocess, sys, torch
maj, mi = torch.version.cuda.split('.')[:2]
m = re.search(r'release (\d+)\.(\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
print('torch CUDA', torch.version.cuda, '| nvcc', m.group(0) if m else 'not found')
if not m or m.group(1) != maj:
    ubuntu = subprocess.run('. /etc/os-release && echo ubuntu${VERSION_ID/./}', shell=True, capture_output=True,
                            text=True, executable='/bin/bash').stdout.strip()
    v = f'{maj}-{mi}'
    !wget -q https://developer.download.nvidia.com/compute/cuda/repos/{ubuntu}/x86_64/cuda-keyring_1.1-1_all.deb && dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null
    !apt-get -qq update > /dev/null && apt-get -qq install -y cuda-minimal-build-{v} cuda-cudart-dev-{v} cuda-cccl-{v} cuda-nvrtc-dev-{v} libcublas-dev-{v} libcusparse-dev-{v} libcusolver-dev-{v} libcurand-dev-{v} > /dev/null
    os.environ['CUDA_HOME'] = f'/usr/local/cuda-{maj}.{mi}'
    os.environ['PATH'] = os.environ['CUDA_HOME'] + '/bin:' + os.environ['PATH']
    !nvcc --version | tail -2

%cd /content
!git clone -q https://github.com/thu-ml/SageAttention.git || true
!git -C SageAttention log -1 --format='SageAttention %h %cd'
%cd /content/SageAttention
# torch >= 2.10 headers use C++20 `requires` clauses; SageAttention's setup.py hardcodes -std=c++17
!sed -i 's/-std=c++17/-std=c++20/g' setup.py && grep -n 'std=c++' setup.py && g++ --version | head -1
!TORCH_CUDA_ARCH_LIST=8.9 EXT_PARALLEL=4 MAX_JOBS=8 pip install --no-build-isolation -v . > /content/sage_build.log 2>&1; tail -40 /content/sage_build.log
%cd /content/video-dit-l4
r = subprocess.run([sys.executable, '-c', 'import importlib.metadata as m, sageattention; print("sageattention", m.version("sageattention"))'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
if r.returncode != 0:
    raise RuntimeError('SageAttention did not build or import; stopping here. Send the output above.')

## Kernel microbenchmark (S2, ~1 min)

In [ ]:
!python bench/attn_micro.py --out results/sage_micro.json

## 33 frames, 4 prompts

Compile with flash attention: this session's baseline (~11 min). The hash check is S1. The first attempt (diffusers 0.43.0) did not reproduce these hashes while fp32 eager did; a match here means the diffusers upgrade caused that.

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --kernels --out results/compile_33f_s3
!python bench/hash_frames.py results/compile_33f_s3 --check results/compile_rerun_33f/frames_sha256.json

Compile with SageAttention (~9 min).

sageattention's Python dispatch calls `torch.cuda.device_count()`, which fullgraph compile cannot trace. `bench/sage_ops.py` wraps the whole call in one `torch.library` custom op, so the compile setup stays the same as the flash run. The first line pulls that fix; the second checks the op against eager `sageattn`, also under fullgraph compile.

In [ ]:
!git pull -q
!python bench/sage_ops.py
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend sage --kernels --out results/sage_33f

In [ ]:
!python bench/compare_frames.py --ref results/compile_33f_s3 --test results/sage_33f

## fp32 reference, fox and market (~28 min), then S6

The .npy files of this run go into the zip, so later experiments can be compared against fp32 without rerunning it.

In [ ]:
!python bench/profile_baseline.py --frames 33 --dtype fp32 --attn efficient --prompt-ids 0 3 --out results/fp32_33f_s3
!python bench/hash_frames.py results/fp32_33f_s3 --check results/fp32_33f/frames_sha256.json

In [ ]:
for t in ['compile_33f_s3', 'sage_33f']:
    print('==', t)
    !python bench/compare_frames.py --ref results/fp32_33f_s3 --test results/{t} --out-name compare_vs_fp32.json

## 81 frames, fox (~17 min), S3, S4 and S8

In [ ]:
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --compile blocks --kernels --out results/compile_81f_s3

In [ ]:
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --compile blocks --attn-backend sage --kernels --out results/sage_81f

In [ ]:
!python bench/compare_frames.py --ref results/compile_81f_s3 --test results/sage_81f

## Optional: INT8 QK with fp16 PV (S7, ~10 min)

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --kernels --out results/sage_fp16pv_33f
!python bench/compare_frames.py --ref results/compile_33f_s3 --test results/sage_fp16pv_33f
!python bench/compare_frames.py --ref results/fp32_33f_s3 --test results/sage_fp16pv_33f --out-name compare_vs_fp32.json

In [ ]:
# Every run's hashes; then zip the new runs without .npy, except the fp32 reference (~80 MB)
import glob, os
runs = sorted(d for d in glob.glob('results/*_s3') + glob.glob('results/sage_*') if os.path.isdir(d))
for d in runs:
    if glob.glob(d + '/frames_*.npy'):
        !python bench/hash_frames.py {d}
runs_str = ' '.join(runs)
!zip -qr sage_results.zip {runs_str} results/sage_micro.json /content/sage_build.log -x '*.npy' && zip -qr sage_results.zip results/fp32_33f_s3/*.npy && ls -lh sage_results.zip
from google.colab import files; files.download('sage_results.zip')